In [1]:
%load_ext autoreload
%autoreload 2

# CICE + MOM6 regional case setup

Sets up a real CrocoDash case with a CICE-coupled compset. CICE's forcing is
now a single expanded-grid restoring file (see `extract_forcings/cice.py`'s
`process_cice_forcing`): the case's domain plus a halo (`n_halo_cells` on
every side), built by ESMF-regridding a CICE-shaped forcing product onto
that expanded grid, with the product's own time axis (a snapshot held
constant, for the real global-restart product) carried through -- there's
no separate CICE IC/OBC anymore. This demo uses the fast, deterministic
`reference_ice` product by default (no real restart/grid file needed); swap
in `cice_restart` for the real global CICE restart path. It's fully wired
into `Case.configure_forcings`/`process_forcings()` via `CICEConfigurator`,
so it runs automatically alongside MOM6's own forcings below -- no
follow-up call needed.

## Step 0: Paths and imports

Follows the "Fast CrocoDash Case Setup" pattern from the repo's `CLAUDE.md`.

In [2]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

CESMROOT = Path("~/work/installs/full_regional_cesm").expanduser()
CASENAME = "cice_obc_demo"
CASEROOT = (Path("~/croc_cases") / CASENAME).expanduser()
INPUTDIR = (Path("~/scratch/croc_input") / CASENAME).expanduser()

print(CESMROOT)
print(CASEROOT)
print(INPUTDIR)

/glade/u/home/manishrv/work/installs/full_regional_cesm
/glade/u/home/manishrv/croc_cases/cice_obc_demo
/glade/u/home/manishrv/scratch/croc_input/cice_obc_demo


## Step 1: Grid, topography, vertical grid

A small Bering Sea domain -- high enough latitude that the reference CICE
restart used for CICE OBCs later actually has sea ice there in January, but
not so far north that it needs the polar (EPSG:3995) grid machinery. Flat
bathymetry + uniform vgrid, same as the CLAUDE.md fast-setup pattern -- this
is a wiring demo, not a science case.

In [3]:
grid = Grid(
    lenx=15.0,
    leny=10.0,
    resolution=1.0,
    xstart=-170.0,
    ystart=68.0,
    name="cice_obc_demo",
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)
vgrid = VGrid.uniform(nk=10, depth=1000.0, name="cice_obc_demo")

## Step 2: Create the case

Swap `SICE` for `CICE` in the NYF compset (per
`demos/gallery/notebooks/CrocoDash/features/coupling/add_cice.ipynb`) to get
a real coupled sea-ice model instead of the stub.

In [4]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_CICE_MOM6%REGIONAL_SROF_SGLC_SWAV",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)
print("CICE in compset:", case.cice_in_compset)

INFO:	initialize:Initializing the visualCaseGen system...


INFO:  csp_solver:CspSolver initialized.


Creating case...

Running the create_newcase tool with the following command:

/glade/u/home/manishrv/work/installs/full_regional_cesm/cime/scripts/create_newcase --compset 1850_DATM%NYF_SLND_CICE_MOM6%REGIONAL_SROF_SGLC_SWAV --res USER_RES --case /glade/u/home/manishrv/croc_cases/cice_obc_demo --machine derecho --run-unsupported --project ncgd0011 --non-local 



The create_newcase command was successful.

Ocean grid directory /glade/u/home/manishrv/scratch/croc_input/cice_obc_demo/ocnice does not exist.


./xmlchange MOM6_MEMORY_MODE=dynamic_symmetric --non-local



The following additional configuration options are required to run and must be provided with any listed arguments in configure_forcings:
  - CICE: no arguments
  - conditions: boundaries, product_name, function_name, compset
CICE in compset: True


## Step 3: Configure + process forcings

`configure_forcings` sets up MOM6's IC/OBC (pointed at the fast,
deterministic `reference_ocean` product) plus CICE's forcing product --
`CICEConfigurator`'s `cice_product_name`/`cice_function_name`, required
since CICE is in the compset -- pointed at `reference_ice`. Both keep the
whole cell running in seconds on any machine, with no real data
dependency. `process_forcings()` then runs the full driver workflow for
everything active for this compset, including CICE's `process_cice_forcing`.

In [5]:
# reference_ice: fast, deterministic synthetic CICE forcing -- generates
# its own grid, no real CICE restart/grid file needed. Swap these two
# kwargs out (or pass cice_product_name="cice_restart" plus
# cice_function_args={"restart_path": ..., "grid_path": ...}) to use a real
# global CICE restart instead.
#
# reference_ocean: fast, deterministic synthetic MOM6 IC/OBC data -- the
# default GLORYS-via-RDA product returns an empty dataset for this small
# domain/date combination on this system, so this keeps the whole demo
# runnable end to end with no real data dependency at all. Drop
# product_name/function_name (or set them to "GLORYS"/
# "get_glorys_data_script_for_cli") to use real GLORYS instead.
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data",
    cice_product_name="reference_ice",
    cice_function_name="get_reference_ice_data",
    n_halo_cells=2,
)

INFO:CrocoDash.forcing_configurations.base:[SKIP] tides missing args: ['tpxo_elevation_filepath', 'tpxo_velocity_filepath', 'tidal_constituents']


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGC incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[SKIP] WW3 incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[REQUIRED] Activating CICE


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIC incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIronForcing incompatible with compset


... (5 outputs left out)

INFO:CrocoDash.forcing_configurations.base:Configuring conditions


Adding parameter changes to user_nl_cice:

  ice_ic = 'default'

Adding parameter changes to user_nl_cice:

  ns_boundary_type = 'zero_gradient'

Adding parameter changes to user_nl_cice:

  ew_boundary_type = 'zero_gradient'

Adding parameter changes to user_nl_cice:

  close_boundaries = .false.

Adding parameter changes to user_nl_cice:

  advect = 'upwind'

Adding parameter changes to user_nl_cice:

  restore_ice = .true.

Adding parameter changes to user_nl_cice:

  trestore = 90

Adding parameter changes to user_nl_mom:

  ! Initial conditions
  INIT_LAYERS_FROM_Z_FILE = True
  Z_INIT_ALE_REMAPPING = True
  TEMP_SALT_INIT_VERTICAL_REMAP_ONLY = True
  DEPRESS_INITIAL_SURFACE = True
  VELOCITY_CONFIG = file
  TEMP_SALT_Z_INIT_FILE = init_tracers.nc
  SURFACE_HEIGHT_IC_FILE = init_eta.nc
  VELOCITY_FILE = init_vel.nc
  Z_INIT_FILE_PTEMP_VAR = temp
  Z_INIT_FILE_SALT_VAR = salt
  SURFACE_HEIGHT_IC_VAR = eta_t
  U_IC_VAR = u
  V_IC_VAR = v

  ! Open boundary conditions
  OBC_NUMBER_OF

./xmlchange STOP_OPTION=ndays --non-local



./xmlchange STOP_N=8 --non-local



In [6]:
case.process_forcings()

INFO:CrocoDash.extract_forcings.obc:GET [south]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [north]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [west]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [east]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:REGRID [south]: 9-day slices


No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping


... (21 outputs left out)

Setting up Initial Conditions
Regridding Velocities... Done.
Regridding Tracers... Done.
Regridding Free surface... Done.
Saving outputs... 

INFO:CrocoDash.extract_forcings.mom6:...end mom6_forge fill.


INFO:CrocoDash.extract_forcings.ic:Successfully retrieved REFERENCE_OCEAN initial condition data located in /glade/u/home/manishrv/scratch/croc_input/cice_obc_demo/ocean directory.


--- xESMF Regridding Info ---
  Source:        time=9, ncat=1, nj=29, ni=39  (10,179 points)  [0.26 MB]
  Destination:   ny=14, nx=19  (266 points)  [0.00 MB]
  Method:        nearest_s2d
  Weights:       generating new weights
  Periodic:      False
  Locstream out: False
-----------------------------
Generating regridding weights using xESMF with method 'nearest_s2d'...
--- xESMF Regridding Info ---
  Source:        time=9, nj=29, ni=39  (10,179 points)  [0.18 MB]
  Destination:   ny=14, nx=19  (266 points)  [0.00 MB]
  Method:        nearest_s2d
  Weights:       generating new weights
  Periodic:      False
  Locstream out: False
-----------------------------
Generating regridding weights using xESMF with method 'nearest_s2d'...
[timing] bc: 21.7s  ic: 1.0s  cice: 0.3s  total: 23.0s
Case is ready to be built: /glade/u/home/manishrv/croc_cases/cice_obc_demo


## Case is set up

MOM6's own IC/OBC are in `case.inputdir / "ocean"` and CICE's
`cice_forcing.nc` (covering the domain plus its 2-cell halo) is in
`case.inputdir / "sea_ice"` -- both generated by the single `process_forcings()`
call above.

In [7]:
print("caseroot:", case.caseroot)
print("inputdir:", case.inputdir)
print("supergrid_path:", case.supergrid_path)

caseroot: /glade/u/home/manishrv/croc_cases/cice_obc_demo
inputdir: /glade/u/home/manishrv/scratch/croc_input/cice_obc_demo
supergrid_path: /glade/u/home/manishrv/scratch/croc_input/cice_obc_demo/ocean/ocean_hgrid_cice_obc_demo_a3fa7b.nc


## Inspect CICE's forcing file

Confirms the output covers the domain plus its halo, spans the case's date
range, and has nonzero synthetic sea ice -- `reference_ice` tapers
concentration from 0 at the bbox's equatorward edge to 1 at its poleward
edge, so this Bering Sea domain (68-78N) should show partial-to-full
concentration, not real observed ice.

In [ ]:
import xarray as xr

cice_ds = xr.open_dataset(case.inputdir / "sea_ice" / "cice_forcing.nc")
print("dims:", dict(cice_ds.sizes))
print("domain (T-cells):", grid.tlon.shape, "-> with halo:", (cice_ds.sizes["ny"], cice_ds.sizes["nx"]))
print("time:", cice_ds["time"].values)
print("total sea ice area (aicen sum):", float(cice_ds["aicen"].sum()))